# CS-320 · YOLOv1 project · Hassaan — Part B: VOC 2007 loader · NMS · mAP@0.5
**This week's task (due 20 Nov):** *VOC 2007 loader* and *mAP@0.5 and NMS*, used by Ayesha's YOLOv1 reproduction (7×7×30 output, 448 px).

| Section | Content |
|---|---|
| 1 | Download VOC2007 (trainval 5011 + test 4952 images), integrity + class statistics |
| 2 | `VOC2007Detection` loader → image `3×448×448` + target `7×7×25` (`[x,y,w,h,obj,20 classes]`), difficult handling, augmentation, visual check of the encoder |
| 3 | NMS (plain + class-aware) + YOLOv1 output decoder — verified against `torchvision.ops` |
| 4 | mAP@0.5 (VOC07 11-point and all-point), per-class AP, P/R at a score threshold, mAP@[.5:.95] — verified with hand-computed cases |
| 5 | End-to-end sanity on the real VOC test set with synthetic detections (no model needed) |
| 6 | How to evaluate the trained YOLOv1 with one call |

## 0 · Setup

In [ ]:
import os, sys, json, shutil, random
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:                                   # Drive is used to keep results if Colab disconnects
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/yolo_project'
    os.makedirs(DRIVE, exist_ok=True)
except Exception:
    DRIVE = None
print('Drive folder:', DRIVE)

### Project modules
The cells below write the `yolotools` package to disk (same files that live in the GitHub repo under `yolotools/`). Run them once; nothing needs to be edited.

In [ ]:
!mkdir -p /content/cs320-yolov1/yolotools

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/__init__.py
"""CS-320 YOLOv1 project - Hassaan (Data lead) toolkit.

Modules
-------
boxes         box format conversions and IoU
penn_fudan    Penn-Fudan download, PASCAL->YOLO conversion, splits, dataset writer
quality       the 10 dataset quality checks + report
viz           dataset figures (sample grid, mask instances, label check, stats)
augment       box-aware augmentation (HSV, flip, scale+translate)
yolo_dataset  loader for the YOLO-format Penn-Fudan folder
voc2007       PASCAL VOC 2007 download + loader + YOLOv1 7x7xK target encoder
nms           NMS, class-aware NMS, YOLOv1 output decoding
metrics       VOC-style AP / mAP@IoU (07 11-point and all-point), P/R at threshold
evaluate      glue: run a YOLOv1-style model over a dataset and compute mAP
ablation      Ultralytics augmentation on/off ablation helpers
"""
__version__ = "1.0.0"

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/boxes.py
"""Box utilities (NumPy only).

Conventions
-----------
xyxy : (x1, y1, x2, y2)   continuous edge coordinates (x2 > x1)
xywh : (xc, yc, w, h)     centre + size
IoU is computed on continuous coordinates. Because IoU is invariant to scaling each
axis separately, it gives identical values on pixel or on normalised (0-1) boxes.
"""
import numpy as np


def xywh_to_xyxy(b):
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    return np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,
                     b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], axis=1)


def xyxy_to_xywh(b):
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    return np.stack([(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2,
                     b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]], axis=1)


def box_area(b):
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    return np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)


def iou_matrix(a, b):
    """Pairwise IoU between a (N,4) and b (M,4) xyxy boxes -> (N,M)."""
    a = np.asarray(a, dtype=np.float64).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0])
    iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2])
    iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    union = box_area(a)[:, None] + box_area(b)[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-12), 0.0)


def iou_one_to_many(box, boxes):
    return iou_matrix(np.asarray(box).reshape(1, 4), boxes)[0]

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/augment.py
"""Box-aware augmentation for YOLOv1-style loaders (PIL + NumPy).

Follows the YOLOv1 paper: random scale + translation up to 20 %, HSV exposure/saturation jitter up to
x1.5, plus horizontal flip. Boxes are normalised xyxy (0-1) and are transformed together with the image.
"""
import numpy as np
from PIL import Image


def hsv_jitter(img, rng, hue=0.05, sat=1.5, val=1.5):
    hsv = np.array(img.convert("HSV"), dtype=np.float32)
    fs = sat ** rng.uniform(-1, 1)
    fv = val ** rng.uniform(-1, 1)
    hsv[..., 0] = (hsv[..., 0] + rng.uniform(-hue, hue) * 255) % 256
    hsv[..., 1] = np.clip(hsv[..., 1] * fs, 0, 255)
    hsv[..., 2] = np.clip(hsv[..., 2] * fv, 0, 255)
    return Image.fromarray(hsv.astype(np.uint8), "HSV").convert("RGB")


def hflip(img, boxes):
    out = boxes.copy()
    out[:, [0, 2]] = 1.0 - boxes[:, [2, 0]]
    return img.transpose(Image.FLIP_LEFT_RIGHT), out


def scale_translate(img, boxes, rng, max_scale=0.2, max_shift=0.2, min_visible=0.3, min_size=0.01):
    """Return (img, boxes, keep_mask). Boxes losing <min_visible of their area or smaller than min_size are dropped."""
    W, H = img.size
    s = rng.uniform(1 - max_scale, 1 + max_scale)
    tx, ty = rng.uniform(-max_shift, max_shift, 2)
    # output(x', y') samples input at (x'-(0.5+t)W)/s + 0.5W
    coeffs = (1 / s, 0, 0.5 * W - (0.5 + tx) * W / s, 0, 1 / s, 0.5 * H - (0.5 + ty) * H / s)
    img = img.transform((W, H), Image.AFFINE, coeffs, resample=Image.BILINEAR, fillcolor=(127, 127, 127))
    nb = boxes.copy()
    nb[:, [0, 2]] = (boxes[:, [0, 2]] - 0.5) * s + 0.5 + tx
    nb[:, [1, 3]] = (boxes[:, [1, 3]] - 0.5) * s + 0.5 + ty
    area0 = np.clip(nb[:, 2] - nb[:, 0], 0, None) * np.clip(nb[:, 3] - nb[:, 1], 0, None)
    nb = np.clip(nb, 0.0, 1.0)
    area1 = (nb[:, 2] - nb[:, 0]) * (nb[:, 3] - nb[:, 1])
    keep = (area1 >= min_visible * np.maximum(area0, 1e-12)) & ((nb[:, 2] - nb[:, 0]) >= min_size) & ((nb[:, 3] - nb[:, 1]) >= min_size)
    return img, nb, keep


def random_augment(img, boxes, labels, rng=None, p_flip=0.5, p_hsv=1.0, p_affine=1.0,
                   flip=True, hsv=True, affine=True):
    """img: PIL RGB; boxes (N,4) normalised xyxy; labels (N,). Each op can be switched off for ablations."""
    rng = rng or np.random.default_rng()
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4).copy()
    labels = np.asarray(labels).copy()
    if hsv and rng.random() < p_hsv:
        img = hsv_jitter(img, rng)
    if flip and rng.random() < p_flip:
        img, boxes = hflip(img, boxes)
    if affine and rng.random() < p_affine:
        img, boxes, keep = scale_translate(img, boxes, rng)
        boxes, labels = boxes[keep], labels[keep]
    return img, boxes, labels

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/voc2007.py
"""PASCAL VOC 2007: download, XML parsing, loader and YOLOv1 target encoder.

Coordinates: VOC boxes are 1-indexed inclusive pixels -> continuous edges (xmin-1, ymin-1, xmax, ymax),
then normalised to 0-1 by the image size. Images are resized straight to img_size x img_size (as in YOLOv1).
Training excludes `difficult` objects (as darknet's voc_label.py does); evaluation keeps them and flags
them so metrics.py ignores them (official VOC protocol).
"""
import os, subprocess, tarfile, urllib.request
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
from .augment import random_augment
from .boxes import xyxy_to_xywh

try:
    import torch
    from torch.utils.data import Dataset as _Base
except ImportError:
    torch = None
    _Base = object

VOC_CLASSES = ("aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat", "chair", "cow",
               "diningtable", "dog", "horse", "motorbike", "person", "pottedplant", "sheep", "sofa",
               "train", "tvmonitor")
CLASS_TO_ID = {c: i for i, c in enumerate(VOC_CLASSES)}
_OX = "http://host.robots.ox.ac.uk/pascal/VOC/voc2007/"
_PJ = "https://pjreddie.com/media/files/"
TARBALLS = {"trainval": "VOCtrainval_06-Nov-2007.tar", "test": "VOCtest_06-Nov-2007.tar"}
EXPECTED_IMAGES = {"trainval": 5011, "test": 4952, "train": 2501, "val": 2510}


def download_voc2007(dest="/content/data", parts=("trainval", "test")):
    """Download + extract VOC2007 (tries the Oxford host, then a mirror). Returns .../VOCdevkit/VOC2007."""
    os.makedirs(dest, exist_ok=True)
    devkit = os.path.join(dest, "VOCdevkit", "VOC2007")
    for part in parts:
        marker = os.path.join(devkit, "ImageSets", "Main", f"{part}.txt")
        if os.path.exists(marker):
            continue
        tar_path = os.path.join(dest, TARBALLS[part])
        for base in (_OX, _PJ):
            try:
                if subprocess.call(["wget", "-q", "-c", "--tries=3", "-O", tar_path, base + TARBALLS[part]]) == 0 \
                        and os.path.getsize(tar_path) > 1e8:
                    break
            except FileNotFoundError:
                urllib.request.urlretrieve(base + TARBALLS[part], tar_path)
                break
        else:
            raise RuntimeError(f"could not download {TARBALLS[part]}; download it manually into {dest}")
        with tarfile.open(tar_path) as t:
            t.extractall(dest)
        os.remove(tar_path)
    return devkit


def parse_voc_xml(path):
    """-> dict(filename, width, height, objects=[dict(name, label, difficult, truncated, box_edges)])."""
    r = ET.parse(path).getroot()
    size = r.find("size")
    out = {"filename": r.findtext("filename"), "width": int(size.findtext("width")),
           "height": int(size.findtext("height")), "objects": []}
    for o in r.findall("object"):
        bb = o.find("bndbox")
        xmin, ymin, xmax, ymax = (int(float(bb.findtext(k))) for k in ("xmin", "ymin", "xmax", "ymax"))
        out["objects"].append({"name": o.findtext("name").strip(), "label": CLASS_TO_ID[o.findtext("name").strip()],
                               "difficult": int(o.findtext("difficult") or 0),
                               "truncated": int(o.findtext("truncated") or 0),
                               "box_edges": (xmin - 1, ymin - 1, xmax, ymax)})
    return out


def read_image_ids(devkit, image_set):
    with open(os.path.join(devkit, "ImageSets", "Main", f"{image_set}.txt")) as f:
        return [ln.strip() for ln in f if ln.strip()]


def encode_yolov1_target(boxes_xywh, labels, S=7, C=20, collision="first"):
    """-> (S,S,5+C) float32 laid out as [x_off, y_off, w, h, obj, one-hot C] (same as the team's encode_targets).

    One object per cell (YOLOv1 limitation). collision='first' keeps the first object, 'largest' keeps the bigger one.
    """
    t = np.zeros((S, S, 5 + C), dtype=np.float32)
    for (xc, yc, w, h), c in zip(np.asarray(boxes_xywh).reshape(-1, 4), labels):
        j, i = min(int(xc * S), S - 1), min(int(yc * S), S - 1)
        if t[i, j, 4] == 1 and (collision == "first" or w * h <= t[i, j, 2] * t[i, j, 3]):
            continue
        t[i, j, :4] = (xc * S - j, yc * S - i, w, h)
        t[i, j, 4] = 1
        t[i, j, 5:] = 0
        t[i, j, 5 + int(c)] = 1
    return t


def decode_target(t, S=7):
    """Inverse of encode_yolov1_target (for visual checks) -> boxes_xywh (K,4), labels (K,)."""
    ii, jj = np.where(t[..., 4] == 1)
    boxes = np.array([[(j + t[i, j, 0]) / S, (i + t[i, j, 1]) / S, t[i, j, 2], t[i, j, 3]] for i, j in zip(ii, jj)]).reshape(-1, 4)
    return boxes, np.array([int(np.argmax(t[i, j, 5:])) for i, j in zip(ii, jj)], dtype=int)


class VOC2007Detection(_Base):
    """VOC2007 detection dataset.

    __getitem__ -> (image CHW float 0-1, target (S,S,5+C)).  For evaluation use `ground_truth(i)`
    (normalised xyxy + labels + difficult, unaugmented) and `image_tensor(i)`.
    """
    classes = VOC_CLASSES

    def __init__(self, devkit, image_set="trainval", img_size=448, S=7, augment=False,
                 keep_difficult=False, collision="first", aug_kwargs=None):
        self.devkit, self.image_set, self.img_size, self.S = devkit, image_set, img_size, S
        self.augment, self.keep_difficult, self.collision = augment, keep_difficult, collision
        self.aug_kwargs = aug_kwargs or {}
        self.ids = read_image_ids(devkit, image_set)
        self._ann_cache = {}

    def __len__(self):
        return len(self.ids)

    def annotation(self, i):
        if i not in self._ann_cache:
            self._ann_cache[i] = parse_voc_xml(os.path.join(self.devkit, "Annotations", self.ids[i] + ".xml"))
        return self._ann_cache[i]

    def _image(self, i):
        return Image.open(os.path.join(self.devkit, "JPEGImages", self.ids[i] + ".jpg")).convert("RGB")

    def ground_truth(self, i):
        a = self.annotation(i)
        scale = np.array([a["width"], a["height"], a["width"], a["height"]], float)
        e = np.array([o["box_edges"] for o in a["objects"]], float).reshape(-1, 4) / scale
        return {"boxes": e, "labels": np.array([o["label"] for o in a["objects"]], dtype=int),
                "difficult": np.array([bool(o["difficult"]) for o in a["objects"]], dtype=bool)}

    def sample(self, i, rng=None):
        """numpy sample: (uint8 img_size x img_size x 3, boxes_xywh (N,4), labels (N,))."""
        gt = self.ground_truth(i)
        keep = np.ones(len(gt["labels"]), bool) if self.keep_difficult else ~gt["difficult"]
        boxes, labels = gt["boxes"][keep], gt["labels"][keep]
        img = self._image(i)
        if self.augment:
            img, boxes, labels = random_augment(img, boxes, labels, rng, **self.aug_kwargs)
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        return np.asarray(img), xyxy_to_xywh(boxes).astype(np.float32), labels

    def image_tensor(self, i):
        arr = np.asarray(self._image(i).resize((self.img_size, self.img_size), Image.BILINEAR))
        return torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0

    def __getitem__(self, i):
        arr, boxes, labels = self.sample(i)
        target = encode_yolov1_target(boxes, labels, self.S, len(VOC_CLASSES), self.collision)
        return torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0, torch.from_numpy(target)


def dataset_statistics(devkit, image_set):
    """Counts per class (objects / non-difficult objects / images containing the class)."""
    ids = read_image_ids(devkit, image_set)
    obj, nd, imgs = np.zeros(20, int), np.zeros(20, int), np.zeros(20, int)
    for i in ids:
        a = parse_voc_xml(os.path.join(devkit, "Annotations", i + ".xml"))
        seen = set()
        for o in a["objects"]:
            obj[o["label"]] += 1
            nd[o["label"]] += 1 - o["difficult"]
            seen.add(o["label"])
        for c in seen:
            imgs[c] += 1
    return {"images": len(ids), "objects": obj, "non_difficult": nd, "images_with_class": imgs}

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/nms.py
"""Non-Maximum Suppression and YOLOv1 output decoding (NumPy).

NMS rule (same as torchvision.ops.nms): visit boxes in descending score order, keep a
box, and suppress every remaining box whose IoU with it is STRICTLY greater than
`iou_thr`. Ties in score are broken by original index (stable sort).
"""
import numpy as np
from .boxes import iou_one_to_many, xywh_to_xyxy


def to_numpy(x):
    """Accept numpy arrays, lists or torch tensors."""
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    return np.asarray(x)


def nms(boxes, scores, iou_thr=0.5, max_keep=None):
    """Greedy NMS. boxes (N,4) xyxy, scores (N,). Returns kept indices (descending score)."""
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    if len(boxes) != len(scores):
        raise ValueError("boxes and scores must have the same length")
    if not 0.0 <= iou_thr <= 1.0:
        raise ValueError("iou_thr must be in [0, 1]")
    n = len(boxes)
    if n == 0:
        return np.zeros(0, dtype=np.int64)
    order = np.argsort(-scores, kind="stable")
    suppressed = np.zeros(n, dtype=bool)
    keep = []
    for pos in range(n):
        i = order[pos]
        if suppressed[i]:
            continue
        keep.append(i)
        if max_keep is not None and len(keep) >= max_keep:
            break
        rest = order[pos + 1:]
        rest = rest[~suppressed[rest]]
        if len(rest):
            suppressed[rest[iou_one_to_many(boxes[i], boxes[rest]) > iou_thr]] = True
    return np.asarray(keep, dtype=np.int64)


def batched_nms(boxes, scores, labels, iou_thr=0.5, max_keep=None):
    """Class-aware NMS: boxes of different classes never suppress each other."""
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    labels = np.asarray(labels).reshape(-1)
    if len(boxes) == 0:
        return np.zeros(0, dtype=np.int64)
    kept = []
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        kept.append(idx[nms(boxes[idx], scores[idx], iou_thr)])
    kept = np.concatenate(kept)
    kept = kept[np.argsort(-scores[kept], kind="stable")]
    return kept[:max_keep] if max_keep is not None else kept


def decode_yolov1(pred, B=2, C=20, score_thr=0.05, clip=True):
    """Decode ONE image's raw YOLOv1 output (S,S,B*5+C) into candidate detections.

    Layout per cell: [x,y,w,h,conf]*B then C class probabilities, where x,y are offsets
    inside the cell and w,h are fractions of the whole image (as in the team's
    encode_targets / YOLOLoss). Score of box b for class c = conf_b * P(c | object).
    Returns boxes (K,4) xyxy in 0-1 image coordinates, scores (K,), labels (K,).
    """
    pred = to_numpy(pred).astype(np.float64)
    S = pred.shape[0]
    if pred.shape != (S, S, B * 5 + C):
        raise ValueError(f"expected (S,S,{B * 5 + C}), got {pred.shape}")
    rows, cols = np.meshgrid(np.arange(S), np.arange(S), indexing="ij")
    cls_p = pred[..., B * 5:]
    all_boxes, all_scores, all_labels = [], [], []
    for b in range(B):
        x, y, w, h, conf = (pred[..., b * 5 + k] for k in range(5))
        xc, yc = (cols + x) / S, (rows + y) / S
        xyxy = xywh_to_xyxy(np.stack([xc, yc, np.abs(w), np.abs(h)], -1).reshape(-1, 4))
        scores = conf.reshape(-1, 1) * cls_p.reshape(-1, C)          # (S*S, C)
        cell, cls = np.where(scores > score_thr)
        all_boxes.append(xyxy[cell])
        all_scores.append(scores[cell, cls])
        all_labels.append(cls)
    boxes = np.concatenate(all_boxes)
    scores = np.concatenate(all_scores)
    labels = np.concatenate(all_labels).astype(np.int64)
    if clip:
        boxes = np.clip(boxes, 0.0, 1.0)
    return boxes, scores, labels


def postprocess_yolov1(pred, B=2, C=20, score_thr=0.05, iou_thr=0.5, max_det=100):
    """decode_yolov1 + class-aware NMS + top-k. Returns dict(boxes, scores, labels)."""
    boxes, scores, labels = decode_yolov1(pred, B, C, score_thr)
    keep = batched_nms(boxes, scores, labels, iou_thr, max_det)
    return {"boxes": boxes[keep], "scores": scores[keep], "labels": labels[keep]}

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/metrics.py
"""Detection metrics: PASCAL-VOC style AP / mAP at a given IoU (default 0.5).

Matching follows the official VOC devkit
  * detections of a class are processed in descending score order (across all images);
  * each detection is compared with the ground truths of ITS image and class and takes
    the one with the highest IoU; if IoU >= iou_thr:
        - that GT is `difficult`           -> detection ignored (neither TP nor FP)
        - GT not yet matched               -> TP, GT is marked as matched
        - GT already matched               -> FP (duplicate detection)
    otherwise FP;
  * recall's denominator is the number of NON-difficult GTs.
AP is the VOC2007 11-point metric (use_07_metric=True, the official metric for VOC2007)
or the all-point area under the monotone precision envelope (VOC2010+ / use_07_metric=False).

Inputs
------
preds[i] = {"boxes": (N,4) xyxy, "scores": (N,), "labels": (N,)}
gts[i]   = {"boxes": (M,4) xyxy, "labels": (M,), "difficult": (M,) bool (optional)}
Boxes may be pixels or normalised coordinates, as long as preds and gts agree.
"""
import numpy as np
from .boxes import iou_one_to_many


def voc_ap(rec, prec, use_07_metric=True):
    rec, prec = np.asarray(rec, float), np.asarray(prec, float)
    if use_07_metric:
        ap = 0.0
        for t in np.arange(0.0, 1.1, 0.1):
            p = prec[rec >= t].max() if np.any(rec >= t) else 0.0
            ap += p / 11.0
        return float(ap)
    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([0.0], prec, [0.0]))
    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


def match_detections(preds, gts, num_classes, iou_thr=0.5):
    """Greedy VOC matching. Returns (records, npos).

    records: dict of arrays sorted by descending score -
        img, cls, score, status   (status: 1=TP, 0=FP, -1=ignored/difficult)
    npos: (num_classes,) count of non-difficult GTs.
    """
    gt_store, npos = {}, np.zeros(num_classes, dtype=np.int64)
    for i, g in enumerate(gts):
        boxes = np.asarray(g["boxes"], float).reshape(-1, 4)
        labels = np.asarray(g["labels"]).reshape(-1).astype(int)
        diff = np.asarray(g.get("difficult", np.zeros(len(boxes), bool))).reshape(-1).astype(bool)
        for c in np.unique(labels):
            m = labels == c
            gt_store[(i, int(c))] = [boxes[m], diff[m], np.zeros(int(m.sum()), bool)]
            npos[int(c)] += int((~diff[m]).sum())

    img, cls, score, box = [], [], [], []
    for i, p in enumerate(preds):
        b = np.asarray(p["boxes"], float).reshape(-1, 4)
        img.append(np.full(len(b), i)); box.append(b)
        cls.append(np.asarray(p["labels"]).reshape(-1).astype(int))
        score.append(np.asarray(p["scores"], float).reshape(-1))
    if not img:
        z = np.zeros(0)
        return {"img": z.astype(int), "cls": z.astype(int), "score": z, "status": z.astype(int)}, npos
    img, cls, score = np.concatenate(img), np.concatenate(cls), np.concatenate(score)
    box = np.concatenate(box)
    order = np.argsort(-score, kind="stable")
    img, cls, score, box = img[order], cls[order], score[order], box[order]

    status = np.zeros(len(score), dtype=np.int64)
    for k in range(len(score)):
        entry = gt_store.get((int(img[k]), int(cls[k])))
        if entry is None or len(entry[0]) == 0:
            continue                                   # FP
        gboxes, gdiff, gdone = entry
        ious = iou_one_to_many(box[k], gboxes)
        j = int(np.argmax(ious))
        if ious[j] >= iou_thr:
            if gdiff[j]:
                status[k] = -1
            elif not gdone[j]:
                status[k] = 1
                gdone[j] = True
            # else: duplicate of an already matched GT -> stays 0 (FP)
    return {"img": img, "cls": cls, "score": score, "status": status}, npos


def _ap_for_class(status, npos_c, use_07_metric):
    """status: statuses of one class's detections in descending-score order."""
    if npos_c == 0:
        return np.nan, np.zeros(0), np.zeros(0)
    tp = np.cumsum(status == 1)           # ignored (-1) entries add to neither tp nor fp
    fp = np.cumsum(status == 0)
    rec = tp / float(npos_c)
    prec = tp / np.maximum(tp + fp, np.finfo(np.float64).eps)
    return voc_ap(rec, prec, use_07_metric), rec, prec


def evaluate_detections(preds, gts, num_classes, iou_thr=0.5, use_07_metric=True, class_names=None):
    """Per-class AP and mAP at one IoU threshold."""
    recs, npos = match_detections(preds, gts, num_classes, iou_thr)
    ap = np.full(num_classes, np.nan)
    curves, final_p, final_r = [], np.zeros(num_classes), np.zeros(num_classes)
    for c in range(num_classes):
        st = recs["status"][recs["cls"] == c]
        a, r, p = _ap_for_class(st, int(npos[c]), use_07_metric)
        ap[c] = a
        curves.append((r, p))
        if len(r):
            final_r[c], final_p[c] = r[-1], p[-1]
    valid = ~np.isnan(ap)
    return {"mAP": float(ap[valid].mean()) if valid.any() else 0.0, "ap": ap, "npos": npos,
            "curves": curves, "precision": final_p, "recall": final_r, "iou_thr": iou_thr,
            "metric": "VOC07-11pt" if use_07_metric else "all-point",
            "class_names": list(class_names) if class_names is not None else [str(i) for i in range(num_classes)]}


def map_over_iou_thresholds(preds, gts, num_classes, thresholds=None, use_07_metric=False):
    """mAP at several IoU thresholds (default 0.50:0.05:0.95) -> dict(thr -> mAP) + 'mean'."""
    thresholds = np.round(np.arange(0.5, 0.96, 0.05), 2) if thresholds is None else thresholds
    out = {float(t): evaluate_detections(preds, gts, num_classes, float(t), use_07_metric)["mAP"]
           for t in thresholds}
    out["mean"] = float(np.mean([v for k, v in out.items()]))
    return out


def precision_recall_at_threshold(preds, gts, num_classes, score_thr, iou_thr=0.5):
    """Micro-averaged TP/FP/FN, precision, recall, F1 using only detections with score >= score_thr.

    Valid because matching is greedy in descending score: dropping the low-score tail does not
    change how higher-scored detections are matched.
    """
    recs, npos = match_detections(preds, gts, num_classes, iou_thr)
    st = recs["status"][recs["score"] >= score_thr]
    tp, fp = int((st == 1).sum()), int((st == 0).sum())
    fn = int(npos.sum()) - tp
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": p, "recall": r, "f1": f1, "score_thr": score_thr}


def sweep_score_thresholds(preds, gts, num_classes, thresholds, iou_thr=0.5):
    """Convenience for confidence-threshold studies: list of precision_recall_at_threshold."""
    return [precision_recall_at_threshold(preds, gts, num_classes, t, iou_thr) for t in thresholds]


def format_ap_table(result):
    lines = [f"{'class':<14}{'AP':>8}{'GT':>7}"]
    for n, a, k in zip(result["class_names"], result["ap"], result["npos"]):
        lines.append(f"{n:<14}{a * 100:>7.2f}%{int(k):>7}")
    lines.append(f"{'mAP@' + str(result['iou_thr']):<14}{result['mAP'] * 100:>7.2f}%   ({result['metric']})")
    return "\n".join(lines)

In [ ]:
%%writefile /content/cs320-yolov1/yolotools/evaluate.py
"""Glue: run a YOLOv1-style torch model over a dataset -> detections -> mAP.

Works with VOC2007Detection and YoloFolderDataset (both expose image_tensor(i) / ground_truth(i)).
The model must map (N,3,H,W) -> (N,S,S,B*5+C) like the team's YOLOv1 class.
"""
import time
import numpy as np
from .nms import postprocess_yolov1
from .metrics import evaluate_detections, format_ap_table


def predict_dataset(model, dataset, B=2, C=20, device="cuda", batch_size=16, score_thr=0.01,
                    nms_iou=0.5, max_det=100):
    import torch
    model.eval()
    preds, t_model, n = [], 0.0, len(dataset)
    with torch.no_grad():
        for s in range(0, n, batch_size):
            x = torch.stack([dataset.image_tensor(i) for i in range(s, min(n, s + batch_size))]).to(device)
            if device != "cpu":
                torch.cuda.synchronize()
            t0 = time.time()
            out = model(x).float().cpu().numpy()
            if device != "cpu":
                torch.cuda.synchronize()
            t_model += time.time() - t0
            preds += [postprocess_yolov1(o, B, C, score_thr, nms_iou, max_det) for o in out]
    return preds, {"model_ms_per_image": 1000 * t_model / max(n, 1)}


def evaluate_yolov1(model, dataset, B=2, C=20, device="cuda", batch_size=16, score_thr=0.01,
                    nms_iou=0.5, max_det=100, iou_thr=0.5, use_07_metric=True, class_names=None, verbose=True):
    preds, timing = predict_dataset(model, dataset, B, C, device, batch_size, score_thr, nms_iou, max_det)
    gts = [dataset.ground_truth(i) for i in range(len(dataset))]
    res = evaluate_detections(preds, gts, C, iou_thr, use_07_metric, class_names or getattr(dataset, "classes", None))
    res.update(timing)
    if verbose:
        print(format_ap_table(res))
        print(f"model forward: {timing['model_ms_per_image']:.1f} ms/image")
    return res, preds, gts

In [ ]:
import sys
sys.path.insert(0, '/content/cs320-yolov1')
import yolotools
print('yolotools', yolotools.__version__, 'ready')

## 1 · Download VOC 2007 and check it

In [ ]:
from yolotools import voc2007 as voc
DEVKIT = voc.download_voc2007('/content/data')
for s in ['trainval', 'test']:
    n = len(voc.read_image_ids(DEVKIT, s))
    print(f'{s:9s} {n} images (expected {voc.EXPECTED_IMAGES[s]})')
    assert n == voc.EXPECTED_IMAGES[s]

In [ ]:
import pandas as pd
st = {s: voc.dataset_statistics(DEVKIT, s) for s in ['trainval', 'test']}
df = pd.DataFrame({'class': voc.VOC_CLASSES,
                   'trainval_obj': st['trainval']['objects'], 'trainval_imgs': st['trainval']['images_with_class'],
                   'test_obj': st['test']['objects'], 'test_imgs': st['test']['images_with_class'],
                   'test_non_difficult': st['test']['non_difficult']})
print(df.to_string(index=False)); print(df.drop(columns='class').sum())
df.set_index('class')[['trainval_obj', 'test_obj']].plot.bar(figsize=(12, 3.5), title='VOC2007 objects per class'); plt.show()

## 2 · The loader
* `VOC2007Detection(devkit, 'trainval'|'test', img_size=448, S=7, augment=..., keep_difficult=...)`
* `__getitem__` → `(image 3×448×448 float 0-1, target 7×7×25)`; target layout per cell `[x_off, y_off, w, h, obj, one-hot(20)]` — identical to Ayesha's `encode_targets` (so the same loss works with `C=20`; the network output is `7×7×(2·5+20)=7×7×30`).
* Boxes: VOC's 1-indexed inclusive pixels → continuous edges, normalised by image size. Images are resized directly to 448×448 (as in the paper).
* Training excludes `difficult` objects (darknet convention). `ground_truth(i)` returns **all** objects with a `difficult` flag for evaluation.
* One object per cell is a YOLOv1 limitation; on a collision the first object is kept (`collision='largest'` keeps the bigger one).
* `augment=True`: HSV jitter (×1.5), horizontal flip, random scale/translate (±20 %), all box-aware. Evaluation sets must use `augment=False`.

In [ ]:
import torch, matplotlib.patches as patches
from torch.utils.data import DataLoader
tv_plain = voc.VOC2007Detection(DEVKIT, 'trainval', 448)
tv_aug   = voc.VOC2007Detection(DEVKIT, 'trainval', 448, augment=True)
x, t = next(iter(DataLoader(tv_aug, batch_size=8, shuffle=True)))
print('image batch', tuple(x.shape), x.dtype, '| target batch', tuple(t.shape), '| objects in batch', int(t[..., 4].sum()))

def show(ds, ids, title):
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    for ax, i in zip(axes.flat, ids):
        arr, bx, lb = ds.sample(i)
        tgt = voc.encode_yolov1_target(bx, lb, 7, 20)
        db, dl = voc.decode_target(tgt)                       # draw what the TARGET encodes (checks the encoder)
        ax.imshow(arr); ax.axis('off')
        for (xc, yc, w, h), c in zip(db, dl):
            ax.add_patch(patches.Rectangle(((xc - w/2) * 448, (yc - h/2) * 448), w * 448, h * 448, lw=2, ec='lime', fc='none'))
            ax.text((xc - w/2) * 448, (yc - h/2) * 448 - 3, voc.VOC_CLASSES[c], color='black', fontsize=9, bbox=dict(fc='lime', ec='none', pad=1))
        ax.set_title(f'{ds.ids[i]}  ({len(lb)} objs, {len(db)} in target)')
    fig.suptitle(title); plt.tight_layout(); plt.show()
show(tv_plain, range(8), 'No augmentation - boxes decoded from the 7x7x25 target')
show(tv_aug, [0, 1, 2, 3, 0, 1, 2, 3], 'Augmentation ON - the same 4 images twice (different random draws)')

In [ ]:
# how many objects does the one-object-per-cell rule lose? (useful number for the report)
lost = tot = 0
for i in range(0, len(tv_plain), 10):
    _, bx, lb = tv_plain.sample(i)
    tot += len(lb); lost += len(lb) - int(voc.encode_yolov1_target(bx, lb, 7, 20)[..., 4].sum())
print(f'sampled {tot} training objects: {lost} ({100*lost/tot:.1f}%) cannot be encoded (two centres in one 7x7 cell)')

## 3 · Non-Maximum Suppression
`nms(boxes, scores, iou_thr)`: visit boxes by descending score, keep one, suppress every remaining box with IoU **> thr**. `batched_nms` does this per class. `postprocess_yolov1(pred)` decodes one `7×7×30` output (score = box-confidence × class-probability), thresholds and applies class-aware NMS.

In [ ]:
from yolotools.nms import nms, batched_nms, postprocess_yolov1, decode_yolov1
rng = np.random.default_rng(0)
# visual demo: 3 true objects, each detected 4-6 times with jitter
truth = np.array([[20, 20, 100, 120], [130, 40, 200, 140], [60, 150, 150, 230]], float)
boxes, scores = [], []
for t in truth:
    for _ in range(rng.integers(4, 7)):
        boxes.append(t + rng.normal(0, 5, 4)); scores.append(rng.uniform(0.3, 0.99))
boxes, scores = np.array(boxes), np.array(scores)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, thr in zip(axes, [None, 0.3, 0.5, 0.7]):
    keep = np.arange(len(boxes)) if thr is None else nms(boxes, scores, thr)
    for i in keep:
        x1, y1, x2, y2 = boxes[i]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, lw=2, ec='C3', fc='none')); ax.text(x1, y1, f'{scores[i]:.2f}', color='C3', fontsize=8)
    ax.set_xlim(0, 240); ax.set_ylim(250, 0); ax.set_title('raw: %d boxes' % len(boxes) if thr is None else f'NMS IoU>{thr}: {len(keep)} kept')
plt.show()

In [ ]:
# independent verification against torchvision (200 random problems, exact same kept indices)
import torch, torchvision
bad = 0
for seed in range(200):
    g = np.random.default_rng(seed); n = int(g.integers(1, 120))
    xy = g.uniform(0, 100, (n, 2)); wh = g.uniform(1, 40, (n, 2)); b = np.concatenate([xy, xy + wh], 1); s = g.uniform(0, 1, n)
    lab = g.integers(0, 5, n); thr = float(g.choice([0.2, 0.4, 0.5, 0.7]))
    tb, ts, tl = torch.tensor(b), torch.tensor(s), torch.tensor(lab)
    bad += nms(b, s, thr).tolist() != torchvision.ops.nms(tb, ts, thr).tolist()
    bad += batched_nms(b, s, lab, thr).tolist() != torchvision.ops.batched_nms(tb, ts, tl, thr).tolist()
print('mismatches vs torchvision:', bad); assert bad == 0

## 4 · mAP@0.5
`evaluate_detections(preds, gts, num_classes, iou_thr=0.5, use_07_metric=True)` follows the VOC devkit: detections sorted by score; each takes the max-IoU ground truth of its image and class; `IoU ≥ thr` and unmatched → TP, already matched → FP (duplicate), matched to a `difficult` GT → ignored; recall's denominator counts non-difficult GTs only. `use_07_metric=True` is the official VOC2007 **11-point** AP (the number comparable with the YOLO paper's 63.4 %); `False` gives the all-point area.

In [ ]:
from yolotools.metrics import (evaluate_detections, map_over_iou_thresholds, precision_recall_at_threshold,
                               sweep_score_thresholds, format_ap_table)
P = lambda b, s, l: dict(boxes=np.array(b, float).reshape(-1, 4), scores=np.array(s, float), labels=np.array(l))
G = lambda b, l, d=None: dict(boxes=np.array(b, float).reshape(-1, 4), labels=np.array(l), **({} if d is None else dict(difficult=np.array(d))))
# hand-computed case: 3 GT; detections ranked TP, FP, TP
gts = [G([[0, 0, 10, 10], [20, 0, 30, 10], [40, 0, 50, 10]], [0, 0, 0])]
preds = [P([[0, 0, 10, 10], [100, 100, 110, 110], [20, 0, 30, 10]], [.9, .8, .7], [0, 0, 0])]
a = evaluate_detections(preds, gts, 1, 0.5, use_07_metric=False)['mAP']; b = evaluate_detections(preds, gts, 1, 0.5, use_07_metric=True)['mAP']
print('all-point AP', a, '(hand: 1/3 + 1/3*2/3 =', 1/3 + 2/9, ')   11-point AP', b, '(hand:', (4 + 3*2/3)/11, ')')
assert abs(a - (1/3 + 2/9)) < 1e-12 and abs(b - (4 + 2) / 11) < 1e-12
# duplicate -> FP ; difficult -> ignored
assert evaluate_detections([P([[0,0,10,10]]*2, [.9, .8], [0, 0])], [G([[0,0,10,10]], [0])], 1, .5, False)['precision'][0] == 0.5
assert evaluate_detections([P([[0,0,10,10], [20,20,30,30]], [.9, .8], [0, 0])], [G([[0,0,10,10], [20,20,30,30]], [0, 0], [False, True])], 1, .5, False)['mAP'] == 1.0
print('metric unit checks passed')

## 5 · End-to-end sanity on the real VOC test set (synthetic detections, no model)
Build detections from the ground truth: every object is detected with prob. 0.8 with box noise, ~50 % get a duplicate, and ~2 random false positives per image are added. Expected behaviour: ideal detections → mAP = 100 %; noisy → lower; **NMS removes the duplicates and raises mAP**.

In [ ]:
test = voc.VOC2007Detection(DEVKIT, 'test', 448)
gts = [test.ground_truth(i) for i in range(len(test))]
rng = np.random.default_rng(0)

def fake_detections(gts, p_det=0.8, sigma=0.04, p_dup=0.5, n_fp=2):
    out = []
    for g in gts:
        B, S, L = [], [], []
        for b, l in zip(g['boxes'], g['labels']):
            if rng.random() > p_det: continue
            wh = np.array([b[2]-b[0], b[3]-b[1]] * 2)
            for k in range(1 + (rng.random() < p_dup)):
                B.append(b + rng.normal(0, sigma, 4) * wh); S.append(rng.uniform(.5, 1) if k == 0 else rng.uniform(.3, .9)); L.append(l)
        for _ in range(n_fp):
            c = rng.uniform(0.1, 0.8, 2); B.append(np.r_[c, c + rng.uniform(.05, .3, 2)]); S.append(rng.uniform(0, .6)); L.append(rng.integers(0, 20))
        B = np.clip(np.array(B).reshape(-1, 4), 0, 1)
        out.append(dict(boxes=B, scores=np.array(S), labels=np.array(L)))
    return out

perfect = [dict(boxes=g['boxes'], scores=np.ones(len(g['labels'])), labels=g['labels']) for g in gts]
r = evaluate_detections(perfect, gts, 20, 0.5, True, voc.VOC_CLASSES); print('GT used as detections -> mAP', r['mAP']); assert abs(r['mAP'] - 1) < 1e-9
noisy = fake_detections(gts)
after = [dict(zip(('boxes', 'scores', 'labels'), (d['boxes'][k], d['scores'][k], d['labels'][k]))) for d in noisy for k in [batched_nms(d['boxes'], d['scores'], d['labels'], 0.5)]]
for name, d in [('noisy, no NMS', noisy), ('noisy + NMS(0.5)', after)]:
    print(f"{name:18s} mAP@0.5  07-11pt {evaluate_detections(d, gts, 20, .5, True)['mAP']*100:5.2f}%   all-point {evaluate_detections(d, gts, 20, .5, False)['mAP']*100:5.2f}%   #dets {sum(len(x['scores']) for x in d)}")

In [ ]:
res = evaluate_detections(after, gts, 20, 0.5, True, voc.VOC_CLASSES); print(format_ap_table(res))
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].bar(voc.VOC_CLASSES, res['ap']); ax[0].tick_params(axis='x', rotation=90); ax[0].set_title('AP per class (VOC07 11-pt)')
r_, p_ = res['curves'][voc.CLASS_TO_ID['person']]; ax[1].plot(r_, p_); ax[1].set_xlabel('recall'); ax[1].set_ylabel('precision'); ax[1].set_title('PR curve - person'); ax[1].set_ylim(0, 1.05)
m = map_over_iou_thresholds(after, gts, 20); ks = [k for k in m if k != 'mean']; ax[2].plot(ks, [m[k] for k in ks], 'o-'); ax[2].set_title(f"mAP vs IoU threshold (mean .5:.95 = {m['mean']:.3f})"); ax[2].set_xlabel('IoU')
plt.tight_layout(); plt.show()
sw = pd.DataFrame(sweep_score_thresholds(after, gts, 20, [0.0, 0.1, 0.3, 0.5, 0.7, 0.9])); print(sw.round(3).to_string(index=False))

## 6 · Evaluating the trained YOLOv1 (for Ayesha's model)
`evaluate_yolov1` runs the model over a dataset, decodes `7×7×30` outputs, applies NMS and returns mAP. Use a **low** score threshold (0.01) for mAP, and the dataset with `augment=False`.

```python
from yolotools.evaluate import evaluate_yolov1
voc_test = voc.VOC2007Detection(DEVKIT, 'test', 448)                 # augment=False
res, preds, gts = evaluate_yolov1(model, voc_test, B=2, C=20, device='cuda',
                                  score_thr=0.01, nms_iou=0.5, iou_thr=0.5, use_07_metric=True)
print(res['mAP'])
```
For Penn-Fudan (1 class): `YoloFolderDataset(OUT, 'test')` with `C=1` (the model must output `7×7×(2·5+1)`), `use_07_metric=False`.  
The guarded cell below runs only if a variable named `model` exists in the notebook.

In [ ]:
if 'model' in globals():
    from yolotools.evaluate import evaluate_yolov1
    res, preds, gts_ = evaluate_yolov1(model, voc.VOC2007Detection(DEVKIT, 'test', 448), B=2, C=20, device='cuda')
else:
    print("no `model` defined here - paste Ayesha's YOLOv1 class + checkpoint loading above to evaluate it")

**Done with Part B.** Everything above is also covered by the 97 automated tests in `tests/` (`python -m pytest tests`), including a randomised cross-check of the AP code against a classic devkit-style reference implementation.